# Experimento: MLP, fase 6 (critério de parada e paciência)

**Por que mais uma fase.** A meta é o KS, mas o MLP sempre parou pela **perda**. A parada pelo KS apareceu duas vezes, e nenhuma delas foi uma medida limpa:

- **fase 3:** a mesma dobra decidia a parada e media o KS, o que inflou o resultado em +0,009 (venceu 20 de 20 na validação e nada no teste);
- **fase 5:** era um eixo do Optuna, mas o TPE só a sorteou 4 vezes em 100 trials.

A paciência longa (30 e 50) foi medida na fase 5, mas só parando pela perda, e empatou. A combinação de parada pelo KS com paciência longa nunca foi medida. Essa combinação importa porque o KS oscila mais que a perda entre épocas, e com paciência curta ele tende a parar num pico de ruído.

## Pergunta

**Nas melhores versões do placar, parar pelo KS, e com mais paciência, sobe o KS?**

| Eixo | Valores |
|---|---|
| Critério de parada | perda ou KS, os dois medidos nos 20% internos da dobra |
| Paciência | 10, 30 ou 50 |
| Versão de partida | `fase4` (o campeão), `slides_otim_rmsprop` (o maior KS do placar) e `optuna` (outra família: logística, sem oversampling, lote 512) |

São 3 × 2 × 3 = 18 células. As três com perda e paciência 10 são as próprias versões, que já estão no placar e não são treinadas de novo. Todo o resto de cada versão (arquitetura, features, oversampling) fica igual.

**Sem vazamento.** O KS que decide a parada é o dos 20% internos. A rede retreina no treino inteiro da dobra pela mediana das épocas, e a dobra externa só mede.

Cada célula leva dois vereditos pareados: contra o trial 0 e contra a **própria versão de partida**. É o segundo que responde à pergunta.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase6.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"              # branch com o código (a fase 6 está nela)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase6-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
USAR_GPU = True                  # placar em pilha na(s) GPU(s); sem GPU, uma rede por vez em vários processos
DISPOSITIVO = "cpu"              # onde roda modelo.py sem GPU: uma rede pequena por vez rende mais na CPU
N_PROCESSOS = 0                  # 0 = todos os núcleos
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa etapas novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp"), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase6-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if DISPOSITIVO:
    os.environ["CHURN_DEVICE"] = DISPOSITIVO
N_PROCESSOS = N_PROCESSOS or os.cpu_count() or 1
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"
    print("MODO_TESTE: 3 épocas e duas células da grade. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


_ultimo_backup = [time.time()]


def backup_periodico():
    if time.time() - _ultimo_backup[0] > 30 * 60:
        salvar(); _ultimo_backup[0] = time.time()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import comum, modelo as M, fase4 as F4, fase5 as F5, fase6 as F6, pilha as P
import torch

GPUS = P.dispositivos_disponiveis() if USAR_GPU and torch.cuda.is_available() else None
F5.usar_pilha(GPUS)
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | pilha: {GPUS or 'desligada'} | modelo.py: {M.DEVICE} | processos: {N_PROCESSOS}")

# O envio ao GitHub falhou a fase 3 inteira sem ninguém perceber: aqui ele é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=6, processos=N_PROCESSOS, gpus=GPUS, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def logar(**k):
    if wandb_run is not None:
        wandb_run.log(k)


# As linhas do MLP já medidas (resultados/placar.jsonl, versionado no repositório) entram aqui sem treinar de novo
if not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    _orig = os.path.join(REPO_DIR, "resultados")
    with open(os.path.join(PASTA, "placar.jsonl"), "w", encoding="utf-8") as _f:
        for _r in comum.ler_placar(_orig):
            if _r["modelo"] == "mlp":
                _f.write(json.dumps(_r) + "\n")
                os.makedirs(os.path.join(PASTA, "placar_oof"), exist_ok=True)
                shutil.copy(os.path.join(_orig, _r["oof"]), os.path.join(PASTA, _r["oof"]))
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "mlp"}
print(f"Placar do MLP: {len(anteriores)} linhas trazidas de resultados/placar.jsonl")
assert all(b in anteriores for b in F6.BASES), "faltam versões de partida no placar"


In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F5.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
y_treino = F4._dados()[2].values.astype(int)
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões do placar sem sobreposição | assinatura do treino {F5.assinatura()}")


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F5.tabela(linhas, campos).round(4))
    salvar()

# 1. Parar pelo KS ou com mais paciência sobe o KS do placar?

Cada célula passa pela régua da fase 5 (15 dobras, parada nos 20% internos, veredito pareado). Vale olhar três colunas:

- **`vs_base`:** o veredito contra a mesma versão parando pela perda com paciência 10. É ele que diz se a regra de parada ajudou;
- **`epocas`:** a época mediana em que a parada parou;
- **`mse_treino` contra `mse_val`:** se parar mais tarde começou a decorar.

In [ ]:
T0 = ("mlp", "trial0")
GRADE = F6.grade_parada()
if MODO_TESTE:
    GRADE = {r: g for r, g in GRADE.items() if r in F6.BASES or r in ("s6_fase4_ks_p10", "s6_optuna_ks_p30")}
itens = [(r, c, {"vs_trial0": T0} if r == b else {"vs_trial0": T0, "vs_base": ("mlp", b)})
         for r, (b, _, _, c) in sorted(GRADE.items(), key=lambda kv: kv[0] not in F6.BASES)]      # as bases entram antes
t0 = time.time()
linhas = F5.garantir(PASTA, itens, n_jobs=N_PROCESSOS)
print(f"{(time.time() - t0) / 60:.1f} min")
for r in linhas:
    v = r.get("vs_base") or {}
    logar(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"], epocas=r.get("epocas"),
          mse_treino=r.get("mse_treino"), mse_val=r.get("mse_val"), delta_vs_base=v.get("delta"))
tg = F5.tabela(linhas, ("vs_trial0", "vs_base"))
meta = pd.DataFrame([dict(rotulo=r, base=b, parada=pa, paciencia=pc) for r, (b, pa, pc, _) in GRADE.items()])
tg = meta.merge(tg, on="rotulo")
for col, nome in (("ks_kfold", "KS do placar"), ("epocas", "época mediana da parada")):
    print(nome)
    display(tg.pivot(index=["base", "parada"], columns="paciencia", values=col).round(4))
print("Veredito contra a própria versão de partida (perda, paciência 10)")
display(tg.pivot(index=["base", "parada"], columns="paciencia", values="vs_base"))
mostrar(linhas)

# 2. Placar do MLP

Todas as versões do MLP, as antigas e as desta fase, na mesma régua.

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "mlp"]
tab = F5.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_mlp_fase6.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_base") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_base") or {}).get("veredito") == "perdeu"]
print(f"Ganharam da própria versão de partida: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 6

- **Alguma célula ganhou da própria versão de partida?** _…_
- **A parada pelo KS precisou de mais paciência que a perda?** _…_
- **A parada mais tarde aumentou a distância entre o MSE do treino e o da validação?** _…_
- **Muda o campeão do MLP?** Só se ganhar no placar. O teste continua usado uma vez, na configuração final. _…_